# BBM 409 - Programming Assignment 2

**PART 1:** Binary Classification with SVM (30 points)  
**PART 2:** Multiclass Classification (70 points)

### Mustafa Ege 2210356088



## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')

---
# PART 1: BINARY CLASSIFICATION WITH SVM (30 POINTS)
---

## Load the Sonar Dataset from UCI Repository

In [ ]:
print("Loading Sonar dataset...")
sonar = fetch_ucirepo(id=151)
X_sonar = sonar.data.features
y_sonar = sonar.data.targets

In [ ]:
print(X_sonar.info())
print("\n\nX_sonar (features):")
print(X_sonar.head())
print("\ny_sonar (target):")
print(y_sonar.head())

## Encode the Target Labels using LabelEncoder()

In [ ]:
le = LabelEncoder()
y_sonar_encoded = le.fit_transform(y_sonar.iloc[:, 0])

print(f"Original classes: {le.classes_}")
print(f"Encoded classes: {np.unique(y_sonar_encoded)}")

## Split the Data into 80% Training and 20% Testing Sets

In [ ]:
X_train_sonar, X_test_sonar, y_train_sonar, y_test_sonar = train_test_split(
    X_sonar, y_sonar_encoded, test_size=0.2, random_state=17, stratify=y_sonar_encoded
) #stratify helps us making the proportion of classes  in training and test sets roughly the same

print(f"X_train shape: {X_train_sonar.shape}")
print(f"X_test shape: {X_test_sonar.shape}")

---
## 1.1. Linear Kernel SVM without Hyperparameter Tuning (5 points)
---

## Create a Pipeline with StandardScaler and Linear SVM

In [ ]:
linear_svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', random_state=17))
])

## Train the Linear SVM Model

In [ ]:
print("Training Linear SVM...")
linear_svm_pipeline.fit(X_train_sonar, y_train_sonar)
print("Training complete.")

## Predict on the Test Set and Calculate Accuracies and Print it

In [ ]:
y_train_pred_linear = linear_svm_pipeline.predict(X_train_sonar)
y_test_pred_linear = linear_svm_pipeline.predict(X_test_sonar)

In [ ]:
train_accuracy_linear = accuracy_score(y_train_sonar, y_train_pred_linear)
test_accuracy_linear = accuracy_score(y_test_sonar, y_test_pred_linear)

In [ ]:
print(f"Linear SVM Train Accuracy: {train_accuracy_linear:.4f}")
print(f"Linear SVM Test Accuracy: {test_accuracy_linear:.4f}")

## Display Classification Report for Linear SVM

In [ ]:
linear_svm_results = {
    'train_accuracy': train_accuracy_linear,
    'test_accuracy': test_accuracy_linear,
    'report': classification_report(y_test_sonar, y_test_pred_linear, target_names=le.classes_, output_dict=True)
}
print(f"Test Classification Report:\n{classification_report(y_test_sonar, y_test_pred_linear, target_names=le.classes_)}")

## Display Confusion Matrix for Linear SVM

In [ ]:
cm_linear = confusion_matrix(y_test_sonar, y_test_pred_linear)

plt.figure(figsize=(8, 5))
sns.heatmap(cm_linear, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Linear SVM')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

---
## 1.2. SVM with GridSearchCV and 5-Fold Cross-Validation (15 points)
---

## Create a Pipeline for GridSearchCV

In [ ]:
tuned_svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=17))
])

## Define the Parameter Grid for Different Kernels

In [ ]:
C_values = [0.01, 0.1, 1, 10, 100, 1000]
gamma_values = ['scale', 'auto', 0.001, 0.01, 0.1]
degree_values = [2, 3, 4, 5]

In [ ]:
# Parameter grid for Linear Kernel
param_grid_linear = {
    'svm__kernel': ['linear'],
    'svm__C': C_values,
}

# Parameter grid for RBF Kernel
param_grid_rbf = {
    'svm__kernel': ['rbf'],
    'svm__C': C_values,
    'svm__gamma': gamma_values
}

# Parameter grid for Polynomial Kernel
param_grid_poly = {
    'svm__kernel': ['poly'],
    'svm__C': C_values,
    'svm__gamma': gamma_values,
    'svm__degree': degree_values
}

# Combine all parameter grids
param_grid = [param_grid_linear, param_grid_rbf, param_grid_poly]

# Display the total number of combinations to be checked
total_combinations = len(C_values) + (len(C_values) * len(gamma_values)) + (len(C_values) * len(gamma_values) * len(degree_values))
print(f"Total number of parameter combinations to check: {total_combinations}")

## Create Stratified 5-Fold Cross-Validation

In [ ]:
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=17)

## Initialize and Run GridSearchCV on Training Data

In [ ]:
grid_search_svm = GridSearchCV(
    estimator=tuned_svm_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv_stratified,
    verbose=3,
    n_jobs=-1 
)

grid_search_svm.fit(X_train_sonar, y_train_sonar)

## Display Best Parameters and Cross-Validation Score

In [ ]:
# Store the best estimator and result
best_svm_model = grid_search_svm.best_estimator_
tuned_svm_results = grid_search_svm.cv_results_

# Display best parameter and cross validation score
print("--- Best Model Summary ---")
print(f"Best Cross-Validation Score: {grid_search_svm.best_score_:.4f}")
print(f"Best Parameters: {grid_search_svm.best_params_}")

## Evaluate the Best Model on the Test Set

In [ ]:
y_test_pred_tuned = best_svm_model.predict(X_test_sonar)
test_accuracy_tuned = accuracy_score(y_test_sonar, y_test_pred_tuned)
print(f"Tuned SVM Test Accuracy (on unseen data): {test_accuracy_tuned:.4f}")

## Display Top 5 Parameter Combinations

In [ ]:
sorted_indices = tuned_svm_results['mean_test_score'].argsort()[::-1][:5]

top_5_results = pd.DataFrame({
    'Rank': np.arange(1, 6),
    'Mean CV Score': np.array(tuned_svm_results['mean_test_score'])[sorted_indices],
    'Parameters': np.array(tuned_svm_results['params'])[sorted_indices],
})

print("\nTop 5 Parameter Combinations:")
pd.set_option('display.max_colwidth', None)
display(top_5_results)

## Display Classification Report for Tuned SVM

In [ ]:
tuned_svm_report = classification_report(y_test_sonar, y_test_pred_tuned, target_names=le.classes_, output_dict=True)
print("\nTuned SVM Classification Report (Test Set):")
print(classification_report(y_test_sonar, y_test_pred_tuned, target_names=le.classes_))

## Display Confusion Matrix for Tuned SVM

In [ ]:
cm_tuned = confusion_matrix(y_test_sonar, y_test_pred_tuned)

plt.figure(figsize=(8, 5))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix - Tuned SVM (Best Model)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## Compare Linear SVM vs Tuned SVM Results. Compare experimental results. Explain the performance impact of linear and nonlinear kernels. Why is kernel trick important? Also explain why using k-fold cross validation is more advantageous than train-test split. (10 points)

In [ ]:
best_params = grid_search_svm.best_params_
best_cv_score = grid_search_svm.best_score_
best_kernel = best_params.get('svm__kernel', 'N/A')

# Get linear SVM results
linear_test_acc = linear_svm_results['test_accuracy']

print("--- Performance Comparison (Test Set) ---")

comparison_data = {
    'Model': ['Linear SVM (Untuned)', f'Tuned SVM ({best_kernel.upper()} Kernel)'],
    'Test Accuracy': [f"{linear_test_acc:.4f}", f"{test_accuracy_tuned:.4f}"],
    'Best CV Score': ['N/A', f"{best_cv_score:.4f}"],
    'Best Hyperparameters': ['C=1.0 (Default), Kernel=linear', str(best_params)]
}

comparison_df = pd.DataFrame(comparison_data)

print(comparison_df.to_markdown(index=False))

# Store final test accuracies for Part 2 comparison
linear_svm_test_accuracy = comparison_df.iloc[0]['Test Accuracy']
tuned_svm_test_accuracy = comparison_df.iloc[1]['Test Accuracy']

---
# PART 2: MULTICLASS CLASSIFICATION (70 POINTS)
---

## Load the Dry Beans Dataset from UCI Repository

In [ ]:
print("Loading Dry Beans dataset...")
dry_bean = fetch_ucirepo(id=602)
X_beans = dry_bean.data.features
y_beans = dry_bean.data.targets

## Encode the Target Labels using LabelEncoder

## Split the Data into 80% Training and 20% Testing Sets

## Scale the Features using StandardScaler

---
## 2.1. Multinomial Logistic Regression (20 points)
---

## Define the Multinomial Logistic Regression Class

## Define Hyperparameter Grid for Multinomial Logistic Regression

## Run GridSearchCV for Multinomial Logistic Regression

## Evaluate Multinomial Logistic Regression on Test Set

## Display Classification Report for Multinomial Logistic Regression

## Display Confusion Matrix for Multinomial Logistic Regression

---
## 2.2. Decision Tree (20 points)
---

###  Define the Decision Tree model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix. (15 points)

## Define Hyperparameter Grid for Decision Tree

## Run GridSearchCV for Decision Tree

## Display Classification Report for Decision Tree

## Display Confusion Matrix for Decision Tree

###  Describe how a decision tree builds its decision structure. (5 points)

---
## 2.3. XGBoost (20 points)
---

###  Define the XGBoost model. Use GridSearchCV to find the best hyperparameters. Perform multiclass classification on the Dry Beans dataset. Print best hyperparameters, classification report and confusion matrix(10 points)

## Define Hyperparameter Grid for XGBoost

## Run GridSearchCV for XGBoost

## Evaluate XGBoost on Test Set

## Display Classification Report for XGBoost

## Display Confusion Matrix for XGBoost

### What are the revolutionary features of XGBoost compared to other tree-based models? (10 points)

###  Compare the classification results of your Multinomial Logistic Regression, Decision Tree, and XGBoost models on the Dry Beans dataset. Discuss the comparison in terms of overall model performance, risk of overfitting, model complexity and the scenarios in which each model is most effective. (10 points)